# Migrate the original tobacco datasets

Copy paired V1 and V2 images into a neutral, unsplit dataset with source masks and provenance. Choose `error` to stop on conflicting weed masks, `keep_separate` to retain each variant, or `keep_first` to automatically keep the first. Original-sized V1 masks are binary, not weed labels. Enable migration only when ready to create a fresh run.


In [1]:
from pathlib import Path
import json
import sys

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'notebooks/reproduction.py').is_file())
sys.path.insert(0, str(ROOT / 'notebooks'))
from reproduction import new_run, capture_environment


In [2]:
RUN_MIGRATION = True
SOURCE = ROOT / 'data/tobacco_original'
RUN_NAME = 'tobacco-image-migration-01'
CONFLICT_POLICY = 'keep_first'
INCLUDE_V1 = False


## Discover and migrate

V2 campaign 1 patch images and masks were numbered in different orders. Correct this only for the verified 72-image, 864-patch layout and fail on unexpected membership. Other campaigns retain their original numbering. V1 contains processed RGB22 variants and binary original-sized masks; set `INCLUDE_V1 = False` to omit V1 pending further quality checks. V2 is inspected first.


In [3]:
def campaign1_mask_names(campaign):
    campaign = Path(campaign)
    image_names = {f'{number}.jpg' for number in range(1, 73)}
    label_names = {f'{number}.png' for number in range(1, 73)}
    patch_names = {f'{number}.png' for number in range(1, 865)}
    for folder, expected, pattern in (
        (campaign / 'Original 1080P images', image_names, '*.jpg'),
        (campaign / 'Labelled 1080P images', label_names, '*.png'),
        (campaign / 'Patch images' / 'data', patch_names, '*.png'),
        (campaign / 'Patch images' / 'mask', patch_names, '*.png'),
    ):
        if not folder.is_dir() or {path.name for path in folder.glob(pattern)} != expected:
            raise ValueError(f'Unrecognized campaign 1 layout: {folder}')
    lexical_numbers = sorted(range(1, 73), key=str)
    return {f'{number}.png': f'{lexical_numbers.index((number - 1) // 12 + 1) * 12 + (number - 1) % 12 + 1}.png'
            for number in range(1, 865)}

def source_pairs(source, include_v1=True):
    source = Path(source)
    version2 = source / 'Tobacco Aerial Dataset V2' / 'Tobacco Aerial Dataset'
    version1 = source / 'Tobacco Aerial Dataset V1' / 'Ready for traintest tobacco data 352x480'
    if not version2.is_dir() or (include_v1 and not version1.is_dir()):
        raise FileNotFoundError('Required dataset root is missing.')
    campaign1_names = campaign1_mask_names(version2 / 'Campaign no. 1')
    folders = []
    for campaign in range(1, 9):
        folder = version2 / f'Campaign no. {campaign}' / ('Patch Images' if campaign == 2 else 'Patch images')
        folders.append((folder / 'data', folder / 'mask', 'three_class'))
    for group in sorted(path for path in version1.iterdir() if path.is_dir()) if include_v1 else []:
        for folder in (group, group / 'test' / 'RGB', group / 'test' / 'RGB22'):
            if (folder / 'data').is_dir():
                folders.append((folder / 'data', folder / 'mask', 'three_class'))
        if (group / 'original').is_dir():
            folders.append((group / 'original', group / 'mask', 'binary'))
    pairs = []
    for images, masks, kind in folders:
        if not images.is_dir() or not masks.is_dir():
            raise FileNotFoundError(f'Missing paired directories: {images}, {masks}')
        image_names = {path.name for path in images.glob('*.png')}
        mask_names = {path.name for path in masks.glob('*.png')}
        if not image_names or image_names != mask_names:
            raise ValueError(f'Image/mask membership mismatch in {images}: '
                             f'missing masks={sorted(image_names - mask_names)}, '
                             f'missing images={sorted(mask_names - image_names)}')
        pairs.extend((images / name, masks / (campaign1_names[name] if images == version2 / 'Campaign no. 1' / 'Patch images' / 'data' else name), kind)
                     for name in sorted(image_names))
    return pairs

def file_hash(path):
    import hashlib
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def inspect_pair(image_path, mask_path, kind):
    import hashlib
    import numpy as np
    from PIL import Image
    with Image.open(image_path) as image:
        rgb = np.asarray(image.convert('RGB'))
    with Image.open(mask_path) as image:
        mask = np.asarray(image)
    if mask.ndim != 2 or mask.shape != rgb.shape[:2]:
        raise ValueError(f'Invalid mask dimensions: {mask_path}')
    allowed = (0, 1, 2) if kind == 'three_class' else (0, 1, 255)
    if not np.isin(mask, allowed).all():
        raise ValueError(f'Unexpected values in {mask_path}; expected {allowed}')
    digest = hashlib.sha256(str(rgb.shape).encode() + rgb.tobytes()).hexdigest()
    weed = hashlib.sha256((mask == 2).tobytes()).hexdigest() if kind == 'three_class' else None
    return rgb.shape[1], rgb.shape[0], digest, weed

def migrate(source, run, conflict_policy='error', include_v1=True):
    import shutil
    if conflict_policy not in ('error', 'keep_separate', 'keep_first'):
        raise ValueError('CONFLICT_POLICY must be error, keep_separate, or keep_first.')
    source, run = Path(source), Path(run)
    (run / 'images').mkdir()
    (run / 'source_masks').mkdir()
    records, seen = [], {}
    for image, mask, kind in source_pairs(source, include_v1):
        width, height, digest, weed = inspect_pair(image, mask, kind)
        origin = {'image': image.relative_to(source).as_posix(),
                  'mask': mask.relative_to(source).as_posix(),
                  'mask_sha256': file_hash(mask), 'mask_type': kind, 'weed_sha256': weed}
        variants = seen.setdefault(digest, {})
        key = (kind, weed)
        if key in variants:
            record = variants[key]
            if (record['mask_type'], record['weed_sha256']) == key:
                record['sources'].append(origin)
            else:
                record.setdefault('discarded_sources', []).append(origin)
            continue
        if variants and conflict_policy == 'error':
            previous = next(iter(variants.values()))
            raise ValueError(f'Conflicting weed annotations for duplicate image: '
                             f"{previous['sources'][0]['image']} and {origin['image']}. "
                             'Use keep_separate to retain both or keep_first to select one.')
        if variants and conflict_policy == 'keep_first':
            record = next(iter(variants.values()))
            variants[key] = record
            record.setdefault('discarded_sources', []).append(origin)
            continue
        image_id = len(records) + 1
        name = f'image_{image_id:06d}.png'
        shutil.copyfile(image, run / 'images' / name)
        shutil.copyfile(mask, run / 'source_masks' / name)
        record = {'id': image_id, 'file_name': name, 'width': width, 'height': height,
                  'rgb_sha256': digest, 'mask_type': kind, 'weed_sha256': weed,
                  'sources': [origin]}
        variants[key] = record
        records.append(record)
    manifest = {'schema_version': 1, 'source_root': str(source.resolve()),
                'conflict_policy': conflict_policy, 'include_v1': include_v1,
                'campaign_1_pairing': 'numeric_images_lexicographic_masks_72x12', 'images': records}
    (run / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    (run / 'summary.json').write_text(json.dumps({'status': 'complete', 'images': len(records),
        'source_pairs': sum(len(record['sources']) + len(record.get('discarded_sources', []))
                            for record in records),
        'discarded_conflict_sources': sum(len(record.get('discarded_sources', []))
                                          for record in records),
        'binary_images': sum(record['mask_type'] == 'binary' for record in records),
        'conflicting_rgb_groups': sum(len(variants) > 1 for variants in seen.values())}, indent=2), encoding='utf-8')
    return manifest


## Create a run

An interrupted or failed run is incomplete. Choose a new run name before retrying; existing runs are never overwritten.


In [4]:
if RUN_MIGRATION:
    migration_run = new_run(ROOT, RUN_NAME)
    config = {'source': str(SOURCE.resolve()), 'run_name': RUN_NAME,
              'conflict_policy': CONFLICT_POLICY, 'include_v1': INCLUDE_V1,
              'notebook_sha256': file_hash(ROOT / 'dataset_notebooks_simple/03_migrate_dataset.ipynb')}
    (migration_run / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
    capture_environment(migration_run / 'environment')
    manifest = migrate(SOURCE, migration_run, CONFLICT_POLICY, INCLUDE_V1)
    print(f"Migrated {len(manifest['images'])} image/label pairs to {migration_run}")
else:
    print('Migration disabled. Set SOURCE, RUN_NAME, and RUN_MIGRATION to enable it.')


Wrote environment metadata to /home/natsusa/academic/tobacco-article/experiments/reproductions/tobacco-image-migration-01/environment
Migrated 2508 image/label pairs to /home/natsusa/academic/tobacco-article/experiments/reproductions/tobacco-image-migration-01
